In [0]:
# ============================================================
# 01) BRONZE: leer superstore.csv y guardarlo como Delta
# - Lee CSV desde ADLS (carpeta bronze)
# - Normaliza nombres de columnas
# - Escribe Delta en /bronze/superstore_bronze
# - Registra tabla en Unity Catalog
# ============================================================

from pyspark.sql import functions as F

# -------- Config simple --------
CATALOG = "asbdevsuperstore"
SCHEMA = "bronze_schema"
TABLE = "superstore_bronze"

BASE = "abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net"
CSV_PATH = f"{BASE}/bronze/superstore.csv"                 # aquí debe estar tu CSV
DELTA_PATH = f"{BASE}/bronze/{TABLE}"                      # carpeta delta de la tabla

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
# 1) Leer CSV (tu dataset a veces viene con encoding latin1)
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .load(CSV_PATH)
)

In [0]:
# 2) Normalizar nombres (para que todo quede con snake_case)
def snake(c: str) -> str:
    return (c.strip().lower()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
            .replace(".", "_"))

df = df_raw.select([F.col(c).alias(snake(c)) for c in df_raw.columns])

In [0]:
# 3) Columnas técnicas (mínimas)
df = df.withColumn("_ingest_ts", F.current_timestamp())

In [0]:
# 4) Escribir como Delta (Bronze)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .save(DELTA_PATH))

In [0]:

# 5) Registrar tabla en Unity Catalog apuntando al LOCATION
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE}
USING DELTA
LOCATION '{DELTA_PATH}'
""")

In [0]:
# 6) Validar
spark.sql(f"SELECT COUNT(*) AS filas FROM {CATALOG}.{SCHEMA}.{TABLE}").show()
display(spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").limit(10))